## Warm-up: re-drill from memory (5 min, no scrolling up, no old notebooks)

**A.** In the blank cell below, load `../data/loans_messy.csv` and type the full **first-look inspection checklist** from memory (7 items). Run it.

**B.** One line each, time AND space, in proper notation (biggest term only):
- P3 Valid Parentheses (stack):
- P4 Best Time to Buy/Sell (one pass, track min):
- P6 First Unique Char (Counter over lowercase letters):

✅ **Model answer**

**A.** The first-look checklist, one call each: `df.shape` (rows, columns), `df.dtypes` / `df.info()` (types, non-null counts), `df.head()` (eyeball values), `df.isna().sum()` (missing per column), `df.duplicated().sum()` (exact duplicates), `df.describe()` (ranges, impossible values, skew: mean vs median), and `df[col].value_counts()` on each text column (spelling/case variants).

**B.**
- P3 Valid Parentheses: **O(n) time** (one pass), **O(n) space** (the stack can hold every character, e.g. `"(((("`).
- P4 Best Time to Buy/Sell: **O(n) time** (one pass), **O(1) space** (two numbers: lowest price, best profit).
- P6 First Unique Char: **O(n) time** (two passes), **O(1) space** (at most 26 letters in the Counter).

In [1]:
import pandas as pd
df = pd.read_csv("../data/loans_messy.csv")
print(df.shape)
df.info()
print(df.isna().sum())
print("duplicates:", df.duplicated().sum())
print(df.describe())
for col in ["region", "employment", "defaulted"]:
    print(df[col].value_counts(dropna=False).head(10))
df.head()

(495, 7)
<class 'pandas.DataFrame'>
RangeIndex: 495 entries, 0 to 494
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   application_id    495 non-null    str    
 1   application_date  495 non-null    str    
 2   region            495 non-null    str    
 3   employment        469 non-null    str    
 4   income            457 non-null    str    
 5   loan_amount       495 non-null    float64
 6   defaulted         495 non-null    str    
dtypes: float64(1), str(6)
memory usage: 27.2 KB
application_id       0
application_date     0
region               0
employment          26
income              38
loan_amount          0
defaulted            0
dtype: int64
duplicates: 15
        loan_amount
count    495.000000
mean   15044.444444
std     8485.811237
min    -5000.000000
25%     7950.000000
50%    14600.000000
75%    22200.000000
max    30000.000000
region
London         120
Leeds          115
Manchester  

,application_id,application_date,region,employment,income,loan_amount,defaulted
0,A10374,2025-12-26,Leeds,Self-Employed,24600,15800.0,1
1,A10267,2025-07-24,Leeds,Unemployed,"£34,600",16800.0,N
2,A10372,2025-05-31,London,employed,56400,16400.0,yes
3,A10161,2025-02-15,Glasgow,employed,38700,14800.0,1
4,A10327,2025-01-23,London,UNEMPLOYED,69100,2100.0,1


## Concept primer: window functions + CTEs

**A window function computes a value per row using other rows, without collapsing them.**
`GROUP BY` turns many rows into one. `OVER (...)` keeps every row and adds a column.

```sql
func(...) OVER (PARTITION BY <group> ORDER BY <sort>)
```
- `PARTITION BY` = "restart for each group" (like groupby). Leave it out = whole table is one window.
- `ORDER BY` inside `OVER` = order used for ranking / running totals / LAG. Separate from the final `ORDER BY`.

| Function | Use it when the question says | Example |
|---|---|---|
| `ROW_NUMBER()` | "the top 1 / latest / first per group" | 1,2,3,4 (ties broken arbitrarily) |
| `RANK()` | "top N, ties share a place" | 1,2,2,4 (gap after tie) |
| `DENSE_RANK()` | "Nth highest distinct value" | 1,2,2,3 (no gap) |
| `LAG(col)` / `LEAD(col)` | "previous / next row", "month-on-month change" | value from row before/after |
| `SUM(col) OVER (... ORDER BY ...)` | "running total", "cumulative" | adds up to the current row |

**Key trap:** window functions run in `SELECT`, which comes *after* `WHERE`.
So `WHERE ROW_NUMBER() OVER (...) = 1` is an error. Compute the rank in a **CTE**, filter in the outer query:

```sql
WITH ranked AS (                          -- CTE = a named temporary result
    SELECT region, customer_id, income,
           ROW_NUMBER() OVER (PARTITION BY region ORDER BY income DESC) AS rn
    FROM customers
)
SELECT * FROM ranked WHERE rn = 1;        -- richest customer per region
```

**Tiny example:** scores `[90, 80, 80, 70]` ordered DESC: ROW_NUMBER `1,2,3,4` · RANK `1,2,2,4` · DENSE_RANK `1,2,2,3`.

## Problem: window functions on the bank database

Write each query as a string, look at it with `run(...)`, then `check(...)`.
**Use exactly the columns and `ORDER BY` asked for.** Floats are compared at 2 dp.
Target: **25 minutes** for Q1-Q4. Note your start time.

In [2]:
import sqlite3, hashlib
import pandas as pd
con = sqlite3.connect("../data/bank.db")

def run(sql):
    """Run a query and show the result as a table."""
    return pd.read_sql(sql, con)

_ANS = {"q1": "cc48dbdb21", "q2": "164d0b5a44", "q3": "d1ea78f66a", "q4": "1afad47f57"}
def check(key, sql):
    """Checks your query's rows (values AND order) against the hidden answer."""
    rows = con.execute(sql).fetchall()
    norm = [tuple(round(x, 2) if isinstance(x, float) else x for x in r) for r in rows]
    ok = hashlib.md5(repr(norm).encode()).hexdigest()[:10] == _ANS[key]
    print(f"{key}: {'CORRECT' if ok else 'WRONG'}  ({len(rows)} rows)")

## Schema map

```
customers (200 rows)                 one row = one customer
  customer_id  PK ─────────┬──────────────────────┐
  first_name, last_name    │                      │
  segment (NULLs), region  │                      │
  annual_income (NULLs)    │                      │
                           │ 1 : many             │ 1 : many
                           ▼                      ▼
accounts (372)                          loans (158)
  account_id   PK                         loan_id      PK      one row = one loan
  customer_id  FK → customers             customer_id  FK → customers
  account_type, balance, status           loan_type, principal, status, credit_score
       │ 1 : many
       ▼
transactions (9,867)                    one row = one transaction
  txn_id       PK
  account_id   FK → accounts
  txn_ts 'YYYY-MM-DD HH:MM:SS', amount (neg = debit), channel, is_fraud
```

10 customers have no accounts; ~15% of accounts have no transactions.
Month from a timestamp: `substr(txn_ts, 1, 7)` → `'2025-03'`. Columns of one table: `run("PRAGMA table_info(loans)")`.

**Q1. Top 1 per group.** For each `loan_type`, return the single **largest loan by `principal`**.
Columns: `loan_type, loan_id, customer_id, principal`. Order by `loan_type`.

In [3]:
q1 = """
WITH ranked AS (
    SELECT loan_type, loan_id, customer_id, principal,
           ROW_NUMBER() OVER (PARTITION BY loan_type ORDER BY principal DESC) AS rn
    FROM loans
)
SELECT loan_type, loan_id, customer_id, principal
FROM ranked
WHERE rn = 1
ORDER BY loan_type
"""
run(q1)

,loan_type,loan_id,customer_id,principal
0,car,5070,102,39330.29
1,mortgage,5151,191,445013.15
2,overdraft,5088,121,2988.71
3,personal,5104,137,24722.35


In [4]:
check("q1", q1)

q1: CORRECT  (4 rows)


**Q2. Top N per group with ties.** For each `region`, find the **top 3 customers by total balance** (sum of `balance` over all their accounts, rounded to 2 dp). Ties share a rank.
Columns: `region, customer_id, total_balance, rnk`. Order by `region`, `rnk`, `customer_id`.

In [5]:
q2 = """
WITH totals AS (
    SELECT c.region, c.customer_id, ROUND(SUM(a.balance), 2) AS total_balance
    FROM customers c
    JOIN accounts a ON a.customer_id = c.customer_id
    GROUP BY c.region, c.customer_id
),
ranked AS (
    SELECT *, RANK() OVER (PARTITION BY region ORDER BY total_balance DESC) AS rnk
    FROM totals
)
SELECT region, customer_id, total_balance, rnk
FROM ranked
WHERE rnk <= 3
ORDER BY region, rnk, customer_id
"""
run(q2)

,region,customer_id,total_balance,rnk
0,Midlands,81,27706.41,1
1,Midlands,117,25767.13,2
2,Midlands,170,22800.10,3
3,North,18,38985.65,1
4,North,92,32891.36,2
5,North,186,16654.36,3
6,Scotland,133,189465.31,1
7,Scotland,74,27740.39,2
8,Scotland,50,21413.91,3
9,South,31,39600.64,1


In [6]:
check("q2", q2)

q2: CORRECT  (15 rows)


**Q3. Month-on-month change.** Total **debit spend** per month (debits only, `amount < 0`, shown as a positive number, rounded to 2 dp; include all transactions, fraud too). Add the previous month's spend and the % change from the previous month (rounded to 1 dp, computed from the rounded spends). First month has NULLs.
Columns: `month, spend, prev_spend, pct_change`. Order by `month`.

In [7]:
q3 = """
WITH monthly AS (
    SELECT substr(txn_ts, 1, 7) AS month, ROUND(SUM(-amount), 2) AS spend
    FROM transactions
    WHERE amount < 0
    GROUP BY substr(txn_ts, 1, 7)
)
SELECT month, spend,
       LAG(spend) OVER (ORDER BY month) AS prev_spend,
       ROUND(100.0 * (spend - LAG(spend) OVER (ORDER BY month))
             / LAG(spend) OVER (ORDER BY month), 1) AS pct_change
FROM monthly
ORDER BY month
"""
run(q3)

,month,spend,prev_spend,pct_change
0,2024-10,120716.14,NaN,NaN
1,2024-11,172650.26,120716.14,43.0
2,2024-12,70727.10,172650.26,-59.0
3,2025-01,59424.33,70727.10,-16.0
4,2025-02,183035.30,59424.33,208.0
5,2025-03,35707.30,183035.30,-80.5
6,2025-04,104463.52,35707.30,192.6
7,2025-05,61864.74,104463.52,-40.8
8,2025-06,104372.78,61864.74,68.7
9,2025-07,102462.95,104372.78,-1.8


In [8]:
check("q3", q3)

q3: CORRECT  (12 rows)


**Q4. Running total.** For accounts **1001 and 1002**, list every transaction with a **running total of `amount` within each account**, in time order (break timestamp ties by `txn_id`). Round the running total to 2 dp.
Columns: `account_id, txn_id, txn_ts, amount, running_total`. Order by `account_id`, `txn_ts`, `txn_id`.

In [9]:
q4 = """
SELECT account_id, txn_id, txn_ts, amount,
       ROUND(SUM(amount) OVER (PARTITION BY account_id ORDER BY txn_ts, txn_id), 2) AS running_total
FROM transactions
WHERE account_id IN (1001, 1002)
ORDER BY account_id, txn_ts, txn_id
"""
run(q4)

,account_id,txn_id,txn_ts,amount,running_total
0,1001,12,2024-10-04 18:33:07,-20.15,-20.15
1,1001,58,2024-10-18 06:29:47,-252.28,-272.43
2,1001,42,2024-11-06 02:10:21,-16.65,-289.08
3,1001,18,2024-11-10 19:02:46,-31.11,-320.19
4,1001,45,2024-11-12 03:16:39,5385.93,5065.74
...,...,...,...,...,...
80,1002,80,2025-07-20 12:36:58,-8.32,625.56
81,1002,70,2025-07-30 08:03:23,-22.94,602.62
82,1002,75,2025-08-04 14:59:22,-82.06,520.56
83,1002,74,2025-09-06 13:56:06,773.00,1293.56


In [10]:
check("q4", q4)

q4: CORRECT  (85 rows)


## Explain it

1. Why can't you put `ROW_NUMBER() ... = 1` straight in `WHERE`? (Use the order of execution.)
2. In Q1, if two loans of the same type tied on principal, what would `ROW_NUMBER` vs `RANK` return?
3. `GROUP BY` vs `PARTITION BY`: one sentence on the difference in output row count.
4. Q3: write the business sentence you'd give a manager about the biggest monthly swing, plus one caveat.

✅ **Model answer**

1. SQL runs `FROM` → `WHERE` → `GROUP BY` → `HAVING` → `SELECT` → `ORDER BY`. Window functions are computed in `SELECT`, **after** `WHERE` has already run, so `WHERE` can't see them. Compute the rank in a CTE (common table expression), then filter in the outer query.
2. `ROW_NUMBER` gives tied rows 1 and 2 (arbitrarily), so `rn = 1` returns **one** of them; `RANK` gives both rank 1, so `rnk = 1` returns **both**.
3. `GROUP BY` collapses each group into **one row**; `PARTITION BY` keeps **every row** and adds the per-group value as a new column.
4. "Debit spend jumped 208% from January to February 2025 (£59k → £183k), then fell 80.5% in March (to £36k)." Caveat: these swings are big enough to suggest a few unusually large transactions (or fraud, which is included) or incomplete months; check the biggest transactions and the number of days of data per month before reading it as a trend.